# CatAPI RAG Inspection

Учебный notebook для инспекции CatAPI RAG в проекте `cat-breed-assistant`.

Данные уже подготовлены заранее и лежат в репозитории, поэтому notebook **не скачивает CatAPI**, **не использует внешние датасеты** и **не чинит Kaggle-зависимости**.

Пайплайн:

```text
catapi_breed_documents.jsonl → catapi_chunks.jsonl → embeddings → ChromaDB → retrieval → RAG prompt → answer
```

## 1. Цель notebook

Цель notebook — руками проверить, какие CatAPI-документы и chunks уже есть в проекте, как они превращаются в embeddings, как попадают в inspection-index ChromaDB, какие chunks достаются retrieval-слоем и какой context попадёт в LLM.

Это inspection notebook: он не меняет backend, Streamlit, Docker и production-индекс.

## 2. Environment setup

Зависимости устанавливаются вне notebook:

```bash
pip install -r requirements.txt
```

Для Kaggle: если зависимости переустанавливались, нужно сделать restart kernel/session.

В notebook нет обязательных `pip install` ячеек. Если `sentence-transformers` или `chromadb` недоступны, соответствующие inspection-секции дадут понятное сообщение.

## 3. Kaggle: clone repository if needed

Если notebook запускается в чистой Kaggle session, сначала нужно загрузить репозиторий в `/kaggle/working/cat-breed-assistant`, потому что processed JSONL лежат внутри проекта.

Ячейка безопасная: если папка уже существует, повторный clone не выполняется и локальные файлы не перетираются.


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/MataNerdy/cat-breed-assistant.git'
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_PROJECT_DIR = KAGGLE_WORKING / 'cat-breed-assistant'

if KAGGLE_WORKING.exists():
    if KAGGLE_PROJECT_DIR.exists():
        print(f'Repository already exists: {KAGGLE_PROJECT_DIR}')
    else:
        print(f'Cloning {REPO_URL} into {KAGGLE_PROJECT_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(KAGGLE_PROJECT_DIR)], check=True)
        print(f'Cloned repository to {KAGGLE_PROJECT_DIR}')
else:
    print('Not running in Kaggle /kaggle/working environment. Skipping clone cell.')

## 4. Setup and paths

Импортируем базовые модули, находим root проекта, проверяем пути и объявляем helper-функции.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import textwrap
from pathlib import Path
from pprint import pprint
from typing import Any

try:
    import pandas as pd
except Exception as exc:
    pd = None
    print(f'pandas is not available, using JSON preview fallback: {exc}')


def find_project_root() -> Path:
    """Find project root in local Jupyter or Kaggle."""
    env_root = os.getenv('CAT_BREED_ASSISTANT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path('/kaggle/working/cat-breed-assistant'),
        Path('/kaggle/working'),
    ])
    for candidate in candidates:
        root = candidate.resolve()
        if (root / 'app.py').exists() and (root / 'data').exists():
            return root
    raise RuntimeError(
        'Project root not found. In Kaggle, clone the repository to '
        '/kaggle/working/cat-breed-assistant or set CAT_BREED_ASSISTANT_ROOT.'
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DOCS_PATH = PROJECT_ROOT / 'data/processed/catapi_breed_documents.jsonl'
CHUNKS_PATH = PROJECT_ROOT / 'data/processed/catapi_chunks.jsonl'
CHROMA_PATH = PROJECT_ROOT / 'data/chroma'
INSPECTION_CHROMA_PATH = PROJECT_ROOT / 'data/chroma_catapi_inspection'
INSPECTION_COLLECTION = 'catapi_breed_chunks_inspection'

print('PROJECT_ROOT:', PROJECT_ROOT)
for label, path in [
    ('PROCESSED_DOCS_PATH', PROCESSED_DOCS_PATH),
    ('CHUNKS_PATH', CHUNKS_PATH),
    ('CHROMA_PATH', CHROMA_PATH),
    ('INSPECTION_CHROMA_PATH', INSPECTION_CHROMA_PATH),
]:
    print(f'{label}: exists={path.exists()} path={path}')


def preview_text(text: Any, n: int = 300) -> str:
    """Return a compact one-line text preview."""
    compact = ' '.join(str(text or '').split())
    return compact if len(compact) <= n else compact[:n].rstrip() + '...'


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSONL rows from path. Return an empty list with a clear message if missing."""
    if not path.exists():
        print(f'File not found: {path}')
        return []
    rows = []
    with path.open('r', encoding='utf-8') as input_file:
        for line_number, line in enumerate(input_file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON on line {line_number} in {path}: {exc}') from exc
    return rows


def show_rows(rows: list[dict[str, Any]], limit: int = 10) -> None:
    """Show rows as a DataFrame when possible, otherwise print JSON previews."""
    limited_rows = list(rows or [])[:limit]
    if pd is not None:
        try:
            display(pd.DataFrame(limited_rows))
            return
        except Exception as exc:
            print(f'pandas display failed, using JSON preview: {exc}')
    for index, row in enumerate(limited_rows, start=1):
        print(f'--- row {index} ---')
        print(json.dumps(row, ensure_ascii=False, indent=2)[:1600])

## 5. Load processed breed documents

Читаем `data/processed/catapi_breed_documents.jsonl`. Если файла нет, сначала выполните:

```bash
python scripts/build_catapi_documents.py
```

In [ ]:
documents = load_jsonl(PROCESSED_DOCS_PATH)

if not documents:
    print('Run scripts/build_catapi_documents.py first')
else:
    unique_breeds = sorted({doc.get('breed_name') for doc in documents if doc.get('breed_name')})
    print('Documents:', len(documents))
    print('Unique breeds:', len(unique_breeds))
    print('First 5 breeds:', unique_breeds[:5])
    print('First document fields:', sorted(documents[0].keys()))
    print('First document metadata:')
    pprint(documents[0].get('metadata', {}))

    document_rows = []
    for doc in documents[:10]:
        metadata = doc.get('metadata') or {}
        document_rows.append({
            'doc_id': doc.get('doc_id'),
            'breed_id': doc.get('breed_id'),
            'breed_name': doc.get('breed_name'),
            'origin': metadata.get('origin'),
            'temperament': metadata.get('temperament'),
            'text_preview': preview_text(doc.get('text'), 220),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'reference_image_id': metadata.get('reference_image_id'),
            'image_url': metadata.get('image_url'),
        })
    show_rows(document_rows, limit=10)

## 6. Load chunks

Читаем `data/processed/catapi_chunks.jsonl`.

В первой версии один chunk = один профиль породы. Это нормально, потому что CatAPI profile короткий.

In [ ]:
chunks = load_jsonl(CHUNKS_PATH)

if not chunks:
    print('Run scripts/build_catapi_chunks.py first')
else:
    chunk_lengths = [len(chunk.get('text', '')) for chunk in chunks]
    unique_chunk_breeds = sorted({chunk.get('breed_name') for chunk in chunks if chunk.get('breed_name')})
    print('Chunks:', len(chunks))
    print('Unique breeds:', len(unique_chunk_breeds))
    print('Chunk length min/mean/max:', min(chunk_lengths), round(sum(chunk_lengths) / len(chunk_lengths), 2), max(chunk_lengths))

    chunk_rows = []
    for chunk in chunks[:10]:
        metadata = chunk.get('metadata') or {}
        chunk_rows.append({
            'chunk_id': chunk.get('chunk_id'),
            'breed_id': chunk.get('breed_id'),
            'breed_name': chunk.get('breed_name'),
            'origin': metadata.get('origin'),
            'text_preview': preview_text(chunk.get('text'), 260),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'reference_image_id': metadata.get('reference_image_id'),
            'image_url': metadata.get('image_url'),
        })
    show_rows(chunk_rows, limit=5)

## 7. Inspect specific breeds

Проверяем, что в данных есть ключевые породы для ручной диагностики.

In [ ]:
def find_breed_profile(breed_name: str) -> dict[str, Any] | None:
    """Find a processed document by breed name."""
    target = breed_name.lower()
    for doc in documents:
        if str(doc.get('breed_name', '')).lower() == target:
            return doc
    return None

specific_breeds = ['British Shorthair', 'Maine Coon', 'Siamese', 'Persian', 'Sphynx']
profile_rows = []
for breed_name in specific_breeds:
    profile = find_breed_profile(breed_name)
    if profile is None:
        print(f'WARNING: breed not found: {breed_name}')
        continue
    metadata = profile.get('metadata') or {}
    profile_rows.append({
        'breed_id': profile.get('breed_id'),
        'breed_name': profile.get('breed_name'),
        'origin': metadata.get('origin'),
        'temperament': metadata.get('temperament'),
        'text_preview': preview_text(profile.get('text'), 420),
        'wikipedia_url': metadata.get('wikipedia_url'),
        'reference_image_id': metadata.get('reference_image_id'),
        'image_url': metadata.get('image_url'),
    })
show_rows(profile_rows, limit=10)

## 8. Embedding sanity check

Используем `sentence-transformers/all-MiniLM-L6-v2`.

По умолчанию `ALLOW_MODEL_DOWNLOAD = False`: notebook сначала пробует загрузить модель локально. Если модели нет в cache, ячейка не падает и объясняет, что делать.

In [ ]:
ALLOW_MODEL_DOWNLOAD = False
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = None
embeddings_available = False

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    embeddings_available = True
    print('Embedding model loaded:', EMBEDDING_MODEL_NAME)
except Exception as exc:
    print('Embedding model not available locally. Set ALLOW_MODEL_DOWNLOAD=True or run the index build script locally.')
    print('Reason:', exc)


def cosine_similarity(left, right) -> float:
    """Compute cosine similarity for two vectors."""
    dot = sum(float(a) * float(b) for a, b in zip(left, right))
    left_norm = math.sqrt(sum(float(a) * float(a) for a in left))
    right_norm = math.sqrt(sum(float(b) * float(b) for b in right))
    if not left_norm or not right_norm:
        return 0.0
    return dot / (left_norm * right_norm)

if embeddings_available and chunks:
    sample_names = {'British Shorthair', 'Maine Coon', 'Sphynx', 'Siamese', 'Persian'}
    sample_chunks = [chunk for chunk in chunks if chunk.get('breed_name') in sample_names][:5]
    sample_texts = [chunk['text'] for chunk in sample_chunks]
    sample_embeddings = embedding_model.encode(sample_texts)
    print('Embedding shape:', sample_embeddings.shape)

    for query in [
        'British Shorthair round face dense coat',
        'Maine Coon large gentle cat',
        'Sphynx hairless cat care',
    ]:
        query_embedding = embedding_model.encode([query])[0]
        scores = []
        for chunk, chunk_embedding in zip(sample_chunks, sample_embeddings):
            scores.append({
                'query': query,
                'breed_name': chunk.get('breed_name'),
                'cosine_similarity': round(cosine_similarity(query_embedding, chunk_embedding), 4),
                'text_preview': preview_text(chunk.get('text'), 180),
            })
        print('\nQUERY:', query)
        show_rows(sorted(scores, key=lambda row: row['cosine_similarity'], reverse=True), limit=5)

## 9. Build Chroma index from chunks

Создаём inspection-index прямо в notebook.

Важно: используется `data/chroma_catapi_inspection`, а не production `data/chroma`, чтобы notebook не портил backend index.

In [ ]:
inspection_collection = None

if not embeddings_available:
    print('Skipping Chroma index build: embedding model is not available.')
elif not chunks:
    print('Skipping Chroma index build: chunks are empty.')
else:
    try:
        import chromadb
    except Exception as exc:
        print('Skipping Chroma index build: chromadb is not available.')
        print('Reason:', exc)
    else:
        INSPECTION_CHROMA_PATH.mkdir(parents=True, exist_ok=True)
        chroma_client = chromadb.PersistentClient(path=str(INSPECTION_CHROMA_PATH))
        existing_names = [collection.name for collection in chroma_client.list_collections()]
        if INSPECTION_COLLECTION in existing_names:
            chroma_client.delete_collection(INSPECTION_COLLECTION)
        inspection_collection = chroma_client.create_collection(
            name=INSPECTION_COLLECTION,
            metadata={'hnsw:space': 'cosine'},
        )

        ids = [chunk['chunk_id'] for chunk in chunks]
        documents_for_chroma = [chunk['text'] for chunk in chunks]
        metadatas = []
        for chunk in chunks:
            metadata = dict(chunk.get('metadata') or {})
            metadata.update({
                'chunk_id': chunk.get('chunk_id'),
                'doc_id': chunk.get('doc_id'),
                'breed_id': chunk.get('breed_id'),
                'breed_name': chunk.get('breed_name'),
                'source': chunk.get('source'),
            })
            metadatas.append({key: ('' if value is None else value) for key, value in metadata.items()})

        all_embeddings = embedding_model.encode(documents_for_chroma, show_progress_bar=True).tolist()
        batch_size = 64
        for start in range(0, len(chunks), batch_size):
            end = start + batch_size
            inspection_collection.add(
                ids=ids[start:end],
                documents=documents_for_chroma[start:end],
                metadatas=metadatas[start:end],
                embeddings=all_embeddings[start:end],
            )
        print('Inspection collection:', INSPECTION_COLLECTION)
        print('Collection count:', inspection_collection.count())

## 10. Manual retrieval from Chroma

Если inspection-index собран, выполняем vector retrieval по английским запросам.

In [ ]:
def retrieve_from_chroma(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """Retrieve top-k chunks from inspection Chroma collection."""
    if inspection_collection is None or embedding_model is None:
        return []
    query_embedding = embedding_model.encode([query])[0].tolist()
    result = inspection_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )
    rows = []
    for rank, (document, metadata, distance) in enumerate(
        zip(result['documents'][0], result['metadatas'][0], result['distances'][0]),
        start=1,
    ):
        metadata = metadata or {}
        rows.append({
            'rank': rank,
            'distance': round(float(distance), 4),
            'breed_name': metadata.get('breed_name'),
            'breed_id': metadata.get('breed_id'),
            'text_preview': preview_text(document, 240),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'image_url': metadata.get('image_url') or metadata.get('reference_image_id'),
            'text': document,
            'metadata': metadata,
        })
    return rows

english_queries = [
    'British Shorthair round face dense coat',
    'Maine Coon large gentle cat',
    'Sphynx hairless cat care',
    'Siamese vocal intelligent cat',
    'Persian long hair grooming',
]

if inspection_collection is None:
    print('Inspection Chroma collection is not available. Run section 8 after making the embedding model available.')
else:
    for query in english_queries:
        print('\nQUERY:', query)
        rows = retrieve_from_chroma(query, top_k=5)
        show_rows([{key: value for key, value in row.items() if key not in {'text', 'metadata'}} for row in rows], limit=5)

## 11. Russian query debug

`all-MiniLM-L6-v2` в первую очередь англоязычная модель, поэтому русские запросы могут работать хуже.

Возможные улучшения:

- multilingual embedding model;
- перевод query на английский;
- breed alias detection;
- hybrid retrieval.

In [ ]:
russian_queries = [
    'британская кошка круглая морда плотная шерсть',
    'мейн кун большой спокойный кот',
    'как ухаживать за сфинксом',
    'сиамская кошка разговорчивая',
    'персидская кошка длинная шерсть уход',
]

if inspection_collection is None:
    print('Inspection Chroma collection is not available. Russian vector retrieval skipped.')
else:
    for query in russian_queries:
        print('\nQUERY:', query)
        rows = retrieve_from_chroma(query, top_k=5)
        show_rows([{key: value for key, value in row.items() if key not in {'text', 'metadata'}} for row in rows], limit=5)

## 12. Simple breed alias retrieval

Для русских запросов полезно сначала определить породу по alias, а потом ограничить context этой породой.

In [ ]:
RU_BREED_ALIASES = {
    'британ': 'British Shorthair',
    'мейн': 'Maine Coon',
    'сфинкс': 'Sphynx',
    'сиам': 'Siamese',
    'перс': 'Persian',
}


def detect_breed_alias(question: str) -> str | None:
    """Detect a breed name from simple Russian aliases."""
    question_lower = question.lower()
    for alias, breed_name in RU_BREED_ALIASES.items():
        if alias in question_lower:
            return breed_name
    return None


def chunks_for_breed(breed_name: str) -> list[dict[str, Any]]:
    """Return chunks matching a breed name."""
    return [chunk for chunk in chunks if chunk.get('breed_name') == breed_name]

alias_examples = russian_queries
alias_rows = []
for question in alias_examples:
    detected = detect_breed_alias(question)
    breed_chunks = chunks_for_breed(detected) if detected else []
    alias_rows.append({
        'question': question,
        'detected_breed': detected,
        'matched_chunks': len(breed_chunks),
        'first_chunk_preview': preview_text(breed_chunks[0].get('text'), 240) if breed_chunks else None,
    })
show_rows(alias_rows, limit=10)

## 13. Hybrid retrieval demo

Учебный hybrid retrieval:

1. Пытаемся определить породу по русскому alias.
2. Если порода найдена, возвращаем chunks этой породы как top context.
3. Если alias не найден, используем vector search.

Это лучше для русских вопросов, где embedding-модель может плохо сопоставлять русский запрос с английским CatAPI-текстом.

In [ ]:
def hybrid_retrieve(question: str, top_k: int = 5) -> dict[str, Any]:
    """Retrieve context using alias filtering first, then vector search."""
    detected_breed = detect_breed_alias(question)
    if detected_breed:
        breed_chunks = chunks_for_breed(detected_breed)[:top_k]
        rows = []
        for rank, chunk in enumerate(breed_chunks, start=1):
            metadata = chunk.get('metadata') or {}
            rows.append({
                'rank': rank,
                'distance': None,
                'breed_name': chunk.get('breed_name'),
                'breed_id': chunk.get('breed_id'),
                'text_preview': preview_text(chunk.get('text'), 240),
                'wikipedia_url': metadata.get('wikipedia_url'),
                'image_url': metadata.get('image_url') or metadata.get('reference_image_id'),
                'text': chunk.get('text'),
                'metadata': metadata,
            })
        return {'detected_breed': detected_breed, 'mode': 'alias_filter', 'chunks': rows}

    rows = retrieve_from_chroma(question, top_k=top_k) if inspection_collection is not None else []
    return {'detected_breed': None, 'mode': 'vector_search', 'chunks': rows}

hybrid_questions = [
    'Чем британские котики отличаются от обычных?',
    'Как ухаживать за сфинксом?',
    'Расскажи про мейн куна',
    'Какая кошка разговорчивая и умная?',
]

for question in hybrid_questions:
    result = hybrid_retrieve(question, top_k=5)
    print('\nQUESTION:', question)
    print('detected_breed:', result['detected_breed'])
    print('mode:', result['mode'])
    show_rows([{key: value for key, value in row.items() if key not in {'text', 'metadata'}} for row in result['chunks']], limit=5)

## 14. Final RAG prompt inspection

Берём вопрос `Чем британские котики отличаются от обычных?`, получаем context через hybrid retrieval и собираем prompt.

In [ ]:
question = 'Чем британские котики отличаются от обычных?'
hybrid_result = hybrid_retrieve(question, top_k=5)
retrieved_context = hybrid_result['chunks']

system_instruction = '\n'.join([
    'Ты дружелюбный ассистент про породы кошек.',
    'Отвечай на русском языке понятно для обычного пользователя.',
    'Используй только retrieved CatAPI context.',
    'Не выдумывай медицинские рекомендации.',
    'Если данных мало, честно скажи, что информации недостаточно.',
])

context_blocks = []
for row in retrieved_context:
    metadata = row.get('metadata') or {}
    context_blocks.append('\n'.join([
        f"Breed: {row.get('breed_name')}",
        f"Breed ID: {row.get('breed_id')}",
        f"Wikipedia: {row.get('wikipedia_url') or metadata.get('wikipedia_url')}",
        f"Text: {row.get('text')}",
    ]))

prompt = (
    f"SYSTEM:\n{system_instruction}\n\n"
    f"RETRIEVED CATAPI CONTEXT:\n{'\n\n---\n\n'.join(context_blocks)}\n\n"
    f"USER QUESTION:\n{question}"
)

print(prompt)

## 15. Mock answer based on retrieved chunks

Без LLM API делаем простой mock answer на основе retrieved context. Цель — глазами увидеть, какие данные попали в ответ.

In [ ]:
def build_mock_answer(question: str, rows: list[dict[str, Any]]) -> str:
    """Build a simple Russian answer from retrieved chunks."""
    if not rows:
        return 'Контекст не найден, поэтому ответ собрать нельзя.'
    first = rows[0]
    facts = []
    for row in rows[:3]:
        facts.append(f"- {row.get('breed_name')}: {preview_text(row.get('text'), 320)}")
    return '\n'.join([
        f'Вопрос: {question}',
        '',
        'В retrieved context попали такие факты:',
        *facts,
        '',
        f"Основной источник: {first.get('breed_name')} ({first.get('breed_id')})",
    ])

mock_answer = build_mock_answer(question, retrieved_context)
print(mock_answer)

## 16. Optional Mistral answer

Если `MISTRAL_API_KEY` есть в environment/Kaggle Secrets, делаем live LLM call. Если ключа нет, не падаем и пропускаем ячейку.

In [ ]:
if not os.getenv('MISTRAL_API_KEY'):
    print('MISTRAL_API_KEY not found, skipping live LLM call')
else:
    from mistralai import Mistral

    client = Mistral(api_key=os.getenv('MISTRAL_API_KEY'))
    response = client.chat.complete(
        model='mistral-small-latest',
        messages=[
            {'role': 'system', 'content': system_instruction},
            {'role': 'user', 'content': prompt},
        ],
    )
    print(response.choices[0].message.content)

## 17. Conclusions

Проверьте значения ниже после выполнения notebook:

- documents loaded: `len(documents)`;
- chunks loaded: `len(chunks)`;
- embeddings worked: `embeddings_available`;
- Chroma retrieval worked: `inspection_collection is not None`;
- Russian retrieval: vector search может быть слабее на русском;
- hybrid retrieval useful: alias detection помогает русским вопросам про конкретную породу.

Next steps:

- connect CatAPI retriever to backend;
- add image URL display in Streamlit;
- add Wikipedia enrichment;
- try multilingual embeddings.

In [ ]:
print('documents loaded:', len(documents))
print('chunks loaded:', len(chunks))
print('embeddings worked:', embeddings_available)
print('chroma retrieval worked:', inspection_collection is not None)
print('hybrid example mode:', hybrid_result['mode'])